In [1]:
import os
import sys
import glob
import datetime
import calendar
# import warnings

import numpy as np
import xarray as xr
import scipy
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt

In [4]:
DIR_CODE = '/glade/u/home/lgchen/ChesROMS/ChesROMS_UMD/preprocess/forcing'
DIR_DATA = '/glade/u/home/lgchen/ChesROMS/ChesROMS_UMD/data/forcing/era5/smallRegion_1hourly'

ref = np.datetime64("1900-01-01T00:00:00")

In [5]:
FN_IN_1H = 'data_stream-oper_stepType-instant.nc'  # input
FN_IN_3H = 'era5_inst_3h_199401.nc'  # output

ds_in_1h = xr.open_dataset(DIR_DATA + '/' + FN_IN_1H)
ds_in_3h = ds_in_1h.isel(valid_time=slice(0, None, 3))

hours = (ds_in_3h.valid_time.values - ref) / np.timedelta64(1, "h")
ds_in_3h = ds_in_3h.assign_coords(valid_time=hours)
ds_in_3h.valid_time.attrs["units"] = "hours since 1900-01-01 00:00:00"
ds_in_3h.valid_time.attrs["calendar"] = "proleptic_gregorian"

ds_in_3h.to_netcdf(DIR_DATA + '/' + FN_IN_3H)

In [5]:
FN_AC_1H = 'data_stream-oper_stepType-accum.nc'
FN_AC_12H = 'era5_accu_1h_increasingTo12h_199401.nc'
FN_AC_3H = 'era5_accu_3h_increasingTo12h_199401.nc'

ds_ac_1h = xr.open_dataset(DIR_DATA + '/' + FN_AC_1H)
# ds_ac_12h = ds_ac_1h.copy()
lst_var = ['tp', 'slhf', 'ssr', 'str', 'sshf', 'ssrd', 'strd', 'e', 'ro', 'ewss', 'nsss']

ntime = ds_ac_1h.dims['valid_time']
period = 12
seeds = range(1, ntime, period)  # 1, 13, 25, ...
blocks = [(s, min(s+period-1, ntime-1)) for s in seeds]
print(blocks)

for var in lst_var:
    arr = ds_ac_1h[var].values
    for seed, end in blocks:
        arr[seed:end+1] = np.cumsum(arr[seed:end+1], axis=0)
    ds_ac_1h[var].values = arr

ds_ac_1h.to_netcdf(DIR_DATA + '/' + FN_AC_12H)

ds_ac_3h = ds_ac_1h.isel(valid_time=slice(0, None, 3))
ds_ac_3h.to_netcdf(DIR_DATA + '/' + FN_AC_3H)

/glade/derecho/scratch/lgchen/tmp/ipykernel_679/4266812507.py:9: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  ntime = ds_ac_1h.dims['valid_time']


[(1, 12), (13, 24), (25, 36), (37, 48), (49, 60), (61, 72), (73, 84), (85, 96), (97, 108), (109, 120), (121, 132), (133, 144), (145, 156), (157, 168), (169, 180), (181, 192), (193, 204), (205, 216), (217, 228), (229, 240), (241, 252), (253, 264), (265, 276), (277, 288), (289, 300), (301, 312), (313, 324), (325, 336), (337, 348), (349, 360), (361, 372), (373, 384), (385, 396), (397, 408), (409, 420), (421, 432), (433, 444), (445, 456), (457, 468), (469, 480), (481, 492), (493, 504), (505, 516), (517, 528), (529, 540), (541, 552), (553, 564), (565, 576), (577, 588), (589, 600), (601, 612), (613, 624), (625, 636), (637, 648), (649, 660), (661, 672), (673, 684), (685, 696), (697, 708), (709, 720), (721, 732), (733, 743)]


In [6]:
FN_AC_1H = 'data_stream-oper_stepType-accum.nc'
FN_AC_1H_INC = 'era5_accu_1h_increasingTo3h_199401.nc'
FN_AC_3H_INC = 'era5_accu_3h_increasingTo3h_199401.nc'

ds_ac_1h = xr.open_dataset(DIR_DATA + '/' + FN_AC_1H)

hours = (ds_ac_1h.valid_time.values - ref) / np.timedelta64(1, "h")
ds_ac_1h = ds_ac_1h.assign_coords(valid_time=hours)
ds_ac_1h.valid_time.attrs["units"] = "hours since 1900-01-01 00:00:00"
ds_ac_1h.valid_time.attrs["calendar"] = "proleptic_gregorian"

# ds_ac_12h = ds_ac_1h.copy()
ds_ac_1h['par'] = 0.48*ds_ac_1h['ssrd']
ds_ac_1h["par"].attrs["long_name"] = "photosynthetically active radiation at surface"
ds_ac_1h["par"].attrs["units"] = "J m**-2"

lst_var = ['tp', 'slhf', 'ssr', 'str', 'sshf', 'ssrd', 'strd', 'e', 'ro', 'ewss', 'nsss', 'par']

ntime = ds_ac_1h.dims['valid_time']
period = 3
seeds = range(1, ntime, period)  # 1, 4, 7, ...
blocks = [(s, min(s+period-1, ntime-1)) for s in seeds]
print(blocks)

for var in lst_var:
    arr = ds_ac_1h[var].values
    for seed, end in blocks:
        arr[seed:end+1] = np.cumsum(arr[seed:end+1], axis=0)
    ds_ac_1h[var].values = arr

for var in lst_var:
    ds_ac_1h[var].values[0, :, :] = 0.5*(ds_ac_1h[var].values[24, :, :] + ds_ac_1h[var].values[48, :, :])

ds_ac_1h.to_netcdf(DIR_DATA + '/' + FN_AC_1H_INC)

ds_ac_3h = ds_ac_1h.isel(valid_time=slice(0, None, 3))
ds_ac_3h.to_netcdf(DIR_DATA + '/' + FN_AC_3H_INC)

/glade/derecho/scratch/lgchen/tmp/ipykernel_18495/2555109160.py:19: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  ntime = ds_ac_1h.dims['valid_time']


[(1, 3), (4, 6), (7, 9), (10, 12), (13, 15), (16, 18), (19, 21), (22, 24), (25, 27), (28, 30), (31, 33), (34, 36), (37, 39), (40, 42), (43, 45), (46, 48), (49, 51), (52, 54), (55, 57), (58, 60), (61, 63), (64, 66), (67, 69), (70, 72), (73, 75), (76, 78), (79, 81), (82, 84), (85, 87), (88, 90), (91, 93), (94, 96), (97, 99), (100, 102), (103, 105), (106, 108), (109, 111), (112, 114), (115, 117), (118, 120), (121, 123), (124, 126), (127, 129), (130, 132), (133, 135), (136, 138), (139, 141), (142, 144), (145, 147), (148, 150), (151, 153), (154, 156), (157, 159), (160, 162), (163, 165), (166, 168), (169, 171), (172, 174), (175, 177), (178, 180), (181, 183), (184, 186), (187, 189), (190, 192), (193, 195), (196, 198), (199, 201), (202, 204), (205, 207), (208, 210), (211, 213), (214, 216), (217, 219), (220, 222), (223, 225), (226, 228), (229, 231), (232, 234), (235, 237), (238, 240), (241, 243), (244, 246), (247, 249), (250, 252), (253, 255), (256, 258), (259, 261), (262, 264), (265, 267), (26